# Select Sentinel2/ECOSTRESS products

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

In [3]:
from sensorsio.mgrs import get_mgrs_tiles_from_roi

In [4]:
from etdataset.provider import Collection
from etdataset.utils import get_bbox_from_roi
from etdataset.api import select
from etdataset.writer import write_matches, write_results

/home/sarrazine/pyenvs/cesbio/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Inputs

To be modified

In [5]:
# ROI
data_path = os.environ["DATA_PATH"]
s2_tile_id = "28PCA"
roi_path = os.path.join(data_path, "zones", "Zone_Senegal_Centre.shp")  # To be changed

In [12]:
# Acquisition dates
min_date = "2023-02-01"  # Format YYYY-MM-DD
max_date = "2023-03-31"  # Format YYYY-MM-DD

In [19]:
# Delta time between two acquisitions
delta_time = "3 days"

In [20]:
# Other criteria to filter products
max_cloud_cover = 25
min_roi_overlap = 33
min_product_overlap = 40

In [21]:
# Outpit directory
output_path = "out"
os.makedirs(output_path, exist_ok=True)

### To obtain a list of MGRS tiles that overlap a ROI with the percentage of surface area covered

In [22]:
roi_bbox, roi_crs = get_bbox_from_roi(roi_path)
mgrs_tiles = get_mgrs_tiles_from_roi(roi_bbox, roi_crs)
mgrs_tiles

,Name,overlap_geometry,geometry,overlap_percentage
0,28PBA,"POLYGON ((-16.76482 14.46621, -16.75975 13.800...","POLYGON ((-17.78282 14.45638, -16.76482 14.466...",22.271238
1,28PBB,"POLYGON ((-16.76412 14.37783, -17.10000 14.374...","POLYGON ((-17.79446 15.35961, -16.77221 15.370...",20.551890
2,28PCA,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120620
3,28PCB,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627789
4,28PDA,"POLYGON ((-15.90000 14.47105, -15.90000 13.800...","POLYGON ((-15.92826 14.47100, -14.90943 14.472...",1.789794
5,28PDB,"POLYGON ((-15.92790 14.38258, -15.93054 15.000...","POLYGON ((-15.93215 15.37519, -14.90905 15.377...",1.780280


### Select products on a MGRS tile

In [23]:
df1, df2, matches = select(
    Collection.LANDSAT,
    Collection.ECOSTRESS,
    min_date,
    max_date,
    tile_id=s2_tile_id,
    delta=delta_time,
    max_cloud_cover=max_cloud_cover,
    min_roi_overlap=min_roi_overlap,
    min_product_overlap=min_product_overlap,
    only_best_match=True
)

INFO:etdataset.api:Products found for LANDSAT in catalog: 26
INFO:etdataset.api:Products found for LANDSAT after ROI filtering: 25


Granules found: 127


INFO:etdataset.api:Products found for ECOSTRESS in catalog: 12
INFO:etdataset.api:Products found for ECOSTRESS after ROI filtering: 12
Matching datasets ...: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 68.39it/s]
INFO:etdataset.selection:Found 8 matches


In [24]:
matches

,Product_name_LANDSAT,Date_LANDSAT,Product_name_ECOSTRESS,Date_ECOSTRESS,overlap
0,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,100.000000
1,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,ECOv002_L2T_LSTE_26616_002_28PCA_20230317T0458...,2023-03-17,100.000000
2,LC08_L2SP_205051_20230303_20230316_02_T1,2023-03-03,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,54.138454
3,LC08_L2SP_205051_20230319_20230324_02_T1,2023-03-19,ECOv002_L2T_LSTE_26616_002_28PCA_20230317T0458...,2023-03-17,53.814279
4,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,ECOv002_L2T_LSTE_26494_015_28PCA_20230309T0806...,2023-03-09,100.000000
5,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26739_017_28PCA_20230325T0150...,2023-03-25,100.000000
6,LC09_L2SP_205051_20230311_20230313_02_T1,2023-03-11,ECOv002_L2T_LSTE_26494_015_28PCA_20230309T0806...,2023-03-09,53.814280
7,LC09_L2SP_205051_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26739_017_28PCA_20230325T0150...,2023-03-25,54.138454


In [25]:
df1

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,EarthExplorer,LANDSAT,None,0,None,LC09_L1TP_205050_20230327_20230328_02_T1,None,"POLYGON ((-16.85552 14.47134, -15.83339 14.471...","POLYGON ((-17.73370 15.49638, -15.60047 15.512...",100.000000
1,LC09_L2SP_205051_20230327_20230329_02_T1,2023-03-27,EarthExplorer,LANDSAT,None,0,None,LC09_L1TP_205051_20230327_20230328_02_T1,None,"POLYGON ((-15.92718 14.07122, -15.92497 13.473...","POLYGON ((-18.04599 14.05379, -15.92718 14.071...",54.132121
2,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,EarthExplorer,LANDSAT,None,2,None,LC08_L1TP_205050_20230319_20230324_02_T1,None,"POLYGON ((-16.85552 14.47134, -15.83339 14.471...","POLYGON ((-17.73932 15.49902, -15.60607 15.515...",100.000000
3,LC08_L2SP_205051_20230319_20230324_02_T1,2023-03-19,EarthExplorer,LANDSAT,None,7,None,LC08_L1TP_205051_20230319_20230324_02_T1,None,"POLYGON ((-15.93274 14.07120, -15.93051 13.473...","POLYGON ((-18.05154 14.05372, -15.93274 14.071...",53.807985
4,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,EarthExplorer,LANDSAT,None,0,None,LC09_L1TP_205050_20230311_20230311_02_T1,None,"POLYGON ((-16.85552 14.47134, -15.83339 14.471...","POLYGON ((-17.73929 15.49632, -15.60607 15.512...",100.000000
5,LC09_L2SP_205051_20230311_20230313_02_T1,2023-03-11,EarthExplorer,LANDSAT,None,0,None,LC09_L1TP_205051_20230311_20230311_02_T1,None,"POLYGON ((-15.93274 14.07120, -15.93051 13.473...","POLYGON ((-18.05154 14.05372, -15.93274 14.071...",53.807986
6,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,EarthExplorer,LANDSAT,None,5,None,LC08_L1TP_205050_20230303_20230316_02_T1,None,"POLYGON ((-16.85552 14.47134, -15.83339 14.471...","POLYGON ((-17.73373 15.49909, -15.60328 15.515...",100.000000
7,LC08_L2SP_205051_20230303_20230316_02_T1,2023-03-03,EarthExplorer,LANDSAT,None,4,None,LC08_L1TP_205051_20230303_20230316_02_T1,None,"POLYGON ((-15.92718 14.07122, -15.92497 13.473...","POLYGON ((-18.04599 14.05379, -15.92718 14.071...",54.132120


In [26]:
df2

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
1,ECOv002_L2T_LSTE_26494_015_28PCA_20230309T0806...,2023-03-09,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
2,ECOv002_L2T_LSTE_26616_002_28PCA_20230317T0458...,2023-03-17,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
3,ECOv002_L2T_LSTE_26739_017_28PCA_20230325T0150...,2023-03-25,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0


### Select products on a ROI

In [32]:
df1, df2, matches = select(
    Collection.LANDSAT,
    Collection.ECOSTRESS,
    min_date,
    max_date,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    delta=delta_time,
    max_cloud_cover=max_cloud_cover,
    min_roi_overlap=min_roi_overlap,
    min_product_overlap=min_product_overlap,
    only_best_match=False
)

INFO:etdataset.api:Products found for LANDSAT in catalog: 31
INFO:etdataset.api:Products found for LANDSAT after ROI filtering: 6


Granules found: 81


INFO:etdataset.api:Products found for ECOSTRESS in catalog: 81
INFO:etdataset.api:Products found for ECOSTRESS after ROI filtering: 25
Matching datasets ...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 66.96it/s]
INFO:etdataset.selection:Found 23 matches


In [33]:
matches

,Product_name_LANDSAT,Date_LANDSAT,Product_name_ECOSTRESS,Date_ECOSTRESS,overlap
0,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26739_017_28PCA_20230325T0150...,2023-03-25,100.0
1,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26739_017_28PCB_20230325T0150...,2023-03-25,100.0
2,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26748_001_28PCB_20230325T1509...,2023-03-25,100.0
3,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26748_001_28PCA_20230325T1509...,2023-03-25,100.0
4,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26800_017_28PCB_20230329T0015...,2023-03-29,100.0
5,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26800_017_28PCA_20230329T0015...,2023-03-29,100.0
6,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26809_001_28PCA_20230329T1333...,2023-03-29,100.0
7,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,ECOv002_L2T_LSTE_26809_001_28PCB_20230329T1333...,2023-03-29,100.0
8,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,ECOv002_L2T_LSTE_26616_002_28PCA_20230317T0458...,2023-03-17,100.0
9,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,ECOv002_L2T_LSTE_26616_002_28PCB_20230317T0458...,2023-03-17,100.0


In [34]:
df1

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,EarthExplorer,LANDSAT,None,0,None,LC09_L1TP_205050_20230327_20230328_02_T1,None,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73370 15.49638, -15.60047 15.512...",100.0
1,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,EarthExplorer,LANDSAT,None,2,None,LC08_L1TP_205050_20230319_20230324_02_T1,None,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73932 15.49902, -15.60607 15.515...",100.0
2,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,EarthExplorer,LANDSAT,None,0,None,LC09_L1TP_205050_20230311_20230311_02_T1,None,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73929 15.49632, -15.60607 15.512...",100.0
3,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,EarthExplorer,LANDSAT,None,5,None,LC08_L1TP_205050_20230303_20230316_02_T1,None,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73373 15.49909, -15.60328 15.515...",100.0


In [35]:
df2

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,ECOv002_L2T_LSTE_26372_016_28PCB_20230301T1116...,2023-03-01,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
1,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
2,ECOv002_L2T_LSTE_26381_003_28PCB_20230302T0034...,2023-03-02,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
3,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
4,ECOv002_L2T_LSTE_26433_014_28PCA_20230305T0940...,2023-03-05,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
5,ECOv002_L2T_LSTE_26433_014_28PCB_20230305T0940...,2023-03-05,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
6,ECOv002_L2T_LSTE_26433_015_28PCB_20230305T0941...,2023-03-05,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
7,ECOv002_L2T_LSTE_26433_015_28PCA_20230305T0941...,2023-03-05,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
8,ECOv002_L2T_LSTE_26494_014_28PCB_20230309T0805...,2023-03-09,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
9,ECOv002_L2T_LSTE_26494_015_28PCB_20230309T0806...,2023-03-09,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361


### Write results

In [36]:
write_matches(matches, os.path.join(output_path,"matches.csv"))
write_results(df1, os.path.join(output_path,f"products_LANDSAT.csv"))
write_results(df2, os.path.join(output_path,f"products_ECOSTRESS.csv"))